<a href="https://colab.research.google.com/github/shashank3110/genAI/blob/main/langraph_multi_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -U langgraph langchain_google_genai

In [ ]:
pip install faiss-cpu langchain-community

In [3]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"]= userdata.get("GOOGLE_API_KEY")

In [6]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, List

In [51]:
# initialize llm
### Initialize LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",  # Using a supported model: gemini-1.5-flash
    temperature=0
)

In [9]:
# build Agent state

class AgentState(TypedDict):
  message : str
  response : str


In [59]:
# build supervisor agent

def supervisor_agent(state:AgentState):
  query = state.get("message")

  prompt = f""" You are a Supervisor agent and you task is to understand user intent from user's query
   and return an appropriate response: if the user wants writing assistance then
   return "author_agent". When user wants calculation and mathematical assitance return "math_agent"

   user query: {query}

   example 1: Help me write an email to my landlord.
   response: "author_agent"

   example 2: help me calculate monthly installment for 100k mortgate at 8% for 30 years.
   reponse: "math_agent"

    """

  response = llm.invoke(prompt)

  state["response"] = response.content.strip()

  return state


# add conditional edge

# def route_supervisor_decision(state:AgentState):
#   response = state.get("response")

#   print(f"In route_supervisor_decision : {response}")

#   if response == "author_agent":
#     return "author_agent"
#   elif response == "math_agent":
#     return "math_agent"
#   else: # default route to supervisor agent
#     return "supervisor_agent"


def route_supervisor_decision(state: AgentState):
    response = state.get("response", "")
    normalized = response.strip().strip('"').strip("'").strip(".").lower()

    print(f"In route_supervisor_decision : {normalized!r}")

    if "author_agent" in normalized:
        return "author_agent"
    elif "math_agent" in normalized:
        return "math_agent"
    else:
        return "supervisor_agent"


# build author agent

def author_agent(state:AgentState):
  query = state.get("message")

  prompt = f"""You are an Author agent and you task is to understand user intent
   and provide literary assistance and return an appropriate response.

   user query: {query}

   Note only return the requested response and nothing else before or after.
   No explanations please.
    """

  response = llm.invoke(prompt)

  state["response"] = response.content.strip()

  return state


# build math agent
def math_agent(state:AgentState):
  query = state.get("message")

  prompt = f""" You are a Math agent and you task is to understand user intent
   and provide calculation assistance and return an appropriate response.
   Use step by step reasoning for calculation.
   But do not returniextra sentences before or after the actual response.

   user query: {query}
    """

  response = llm.invoke(prompt)

  state["response"] = response.content.strip()

  return state



In [56]:
agent_graph = StateGraph(AgentState)

agent_graph.add_node("supervisor_agent", supervisor_agent)
agent_graph.add_node("author_agent", author_agent)
agent_graph.add_node("math_agent", math_agent)

agent_graph.set_entry_point("supervisor_agent")

agent_graph.add_conditional_edges("supervisor_agent", route_supervisor_decision,

                                  {
                                  "author_agent" : "author_agent",
                                  "math_agent" : "math_agent",
                                  "supervisor_agent" : "supervisor_agent" # Loop back to supervisor if no specific agent is chosen
                                  }
                                  )




agent_graph.add_edge("author_agent", END)
agent_graph.add_edge("math_agent", END)

In [60]:
# compile graph
app = agent_graph.compile()



In [61]:
# add recursion limit for safety
config = {"recursion_limit":10}


# batch invocation
result = app.batch([
                     {"message": "what is 5+5?"},
                     {"message": "calculate monthly installment for 80k mortgate at 7.5% for 30 years."},
                     {"message": "help me wite a short email to my landlord that there is no warm water in my apartment since last 3 days and it is very cold?"},

                    ]

                     , config=config)

In route_supervisor_decision : 'author_agent'
In route_supervisor_decision : 'math_agent'
In route_supervisor_decision : 'math_agent'


In [67]:
for res in result:
  print("query:",res["message"])
  print("response:", res["response"])
  print("\n------------------------------------------------\n")

query: what is 5+5?
response: The user is asking for a simple addition calculation.

**Step 1:** Identify the operation. The operation is addition, indicated by the "+" symbol.
**Step 2:** Identify the numbers involved. The numbers are 5 and 5.
**Step 3:** Perform the addition. 5 + 5 = 10.

**Response:**
5 + 5 = 10

------------------------------------------------

query: calculate monthly installment for 80k mortgate at 7.5% for 30 years.
response: Okay, I can help you calculate the monthly installment for your mortgage.

**User Intent:** The user wants to know the monthly payment amount for a mortgage.

**Information Provided:**
*   Principal Loan Amount (P): $80,000
*   Annual Interest Rate (r): 7.5%
*   Loan Term (t): 30 years

**Calculation Steps:**

1.  **Convert Annual Interest Rate to Monthly Interest Rate:**
    *   The annual interest rate is 7.5%.
    *   To get the monthly interest rate, divide the annual rate by 12.
    *   Monthly interest rate (i) = 7.5% / 12 = 0.075 / 1

In [ ]:
# app.invoke( {"message": """help me wite a short email to my landlord that there is
# no warm water in my apartment since last 3 days and it is very cold?"""}, config=config)